# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [ ]:
MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"

SKYFIELD: str = "IROSDummy"
DATA_FITS: str = "baseline_2-50keV_1ks"

RUN_ID: str = 'IROSbenchmrk_detected'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "detected"

E_min: float = 2.0  # [keV]
E_max: float = 5.0  # [keV]

UP_X, UP_Y = 2, 1

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, UP_X, UP_Y)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


CATALOG ='IROS_benchmark_catalogue_1ks.fits' / Catalog file                      [astropy.io.fits.card]


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
from numpy.typing import NDArray

def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    varmap: NDArray,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(
        log, catalogue, sdl, camera,
    )
    cts = np.array(log.log['fluence'])
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    n, m = varmap.shape
    boxsize = (80, 200)
    srows, scols = (
        slice((n - 1) // 2 - camera.upscale_f.y * boxsize[0], (n - 1) // 2 + camera.upscale_f.y * boxsize[0] + 1),
        slice((m - 1) // 2 - camera.upscale_f.x * boxsize[1], (m - 1) // 2 + camera.upscale_f.x * boxsize[1] + 1),
    )
    rmse = np.sqrt(np.mean(varmap[srows, scols]))
    dmap = {
        log.name: {
            'Source': ids,
            'DthetaX': theta_res_x,
            'DthetaY': theta_res_y,
            'IROS_cts': cts,
            'True_cts': true_cts,
            'Dcts': (cts - true_cts) / np.sqrt(true_cts),
            'Dcts_var': (cts - true_cts) / rmse,
            'SNR': np.array(log.log['snr']),
            # 'thetaX [deg]': np.array(log.log['angle_x']),
            # 'thetaY [deg]': np.array(log.log['angle_y']),
        }
    }
    return pd.DataFrame(dmap)

def get_joint_tab(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """Generates a Dataframe with output data from both cameras."""
    compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
    dmap = {
        'Source': np.array(data_camA.CAM1A['Source'])[unitmap.idx_a],

        'DthetaX_A': np.array(data_camA.CAM1A['DthetaX'])[unitmap.idx_a],
        'DthetaY_A': np.array(data_camA.CAM1A['DthetaY'])[unitmap.idx_a],
        'TrueCts_A': np.array(data_camA.CAM1A['True_cts'])[unitmap.idx_a],
        'ReconstrCts_A': np.array(data_camA.CAM1A['IROS_cts'])[unitmap.idx_a],
        'Dcts_A': np.array(data_camA.CAM1A['Dcts'])[unitmap.idx_a],
        'Dcts_A_var': np.array(data_camA.CAM1A['Dcts_var'])[unitmap.idx_a],

        'DthetaX_B': np.array(data_camB.CAM1B['DthetaX'])[unitmap.idx_b],
        'DthetaY_B': np.array(data_camB.CAM1B['DthetaY'])[unitmap.idx_b],
        'TrueCts_B': np.array(data_camB.CAM1B['True_cts'])[unitmap.idx_b],
        'ReconstrCts_B': np.array(data_camB.CAM1B['IROS_cts'])[unitmap.idx_b],
        'Dcts_B': np.array(data_camB.CAM1B['Dcts'])[unitmap.idx_b],
        'Dcts_B_var': np.array(data_camB.CAM1B['Dcts_var'])[unitmap.idx_b],

        'SNR': compose(
            np.array(data_camA.CAM1A['SNR'])[unitmap.idx_a],
            np.array(data_camB.CAM1B['SNR'])[unitmap.idx_b],
        ),
    }
    return pd.DataFrame(dmap)

In [6]:
from bloodmoon.mask import count, variance

def get_varmap(camera: CodedMaskCamera, sdl: DataLoader) -> NDArray:
    detector = count(camera, sdl.DLdata)[0]
    varmap = variance(camera, detector)
    return varmap


varmapA, varmapB = map(lambda sdl: get_varmap(wfm, sdl), (sdlA, sdlB))

In [7]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm, varmapA)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm, varmapB)


Pixel angular resolution at upscaling (x, y): (2, 1)
  - fine direction: 2.1163 arcmin
  - coarse direction: 8.4653 arcmin



Analysing S17:   0%|          | 0/25 [00:00<?, ?it/s]WARNING: The following header keyword is invalid or follows an unrecognized non-standard convention:
CATALOG ='IROS_benchmark_catalogue_1ks.fits' / Catalog file                      [astropy.io.fits.card]
Analysing LEMX-CAM1BS4: 100%|██████████| 25/25 [00:01<00:00, 19.89it/s]


In [ ]:
unit_data = get_joint_tab(data_camA, data_camB, cu_map)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.1f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    save_to=f'{OUT_RESULTS_PATH}/../texTable_Unit_results_{DATASET}_{E_min}-{E_max}.tex',
    overwrite=True,
    **KWS,
)

In [9]:
unit_data.sort_values('SNR', ascending=False, ignore_index=True)

,Source,DthetaX_A,DthetaY_A,TrueCts_A,ReconstrCts_A,Dcts_A,Dcts_A_var,DthetaX_B,DthetaY_B,TrueCts_B,ReconstrCts_B,Dcts_B,Dcts_B_var,SNR
0,S17,-0.070755,-0.218513,386129.0,396380.343351,16.497364,9.012038,0.025429,0.528560,437633.0,448053.389554,15.751754,8.870300,699.172114
1,S29,-0.006766,5.237642,116328.0,125570.479048,27.098568,8.125138,-0.049603,-0.644161,131252.0,142821.696691,31.935154,9.848641,183.030548
2,S25,0.011965,0.476997,110671.0,116235.328962,16.726146,4.891646,0.031082,5.445319,101354.0,93994.561032,-23.116616,-6.264682,113.697431
3,S19,-0.015980,0.237405,72442.0,73921.943701,5.498573,1.301030,0.018326,2.554423,78071.0,83864.751484,20.735523,4.931899,81.774716
4,S11,0.004757,-2.242213,52094.0,47355.646428,-20.760299,-4.165525,-0.025043,1.276502,58923.0,66788.402097,32.402499,6.695381,73.927351
5,S26,-0.062468,2.347232,56604.0,50045.965352,-27.564502,-5.765221,-0.022878,-4.097017,56898.0,53072.979873,-16.035600,-3.256027,69.243251
6,S13,-0.181762,1.843711,47845.0,43293.130598,-20.809973,-4.001585,0.030098,4.799961,49000.0,57591.051435,38.810414,7.313086,63.281924
7,S24,-0.090797,-1.891460,52192.0,44329.484470,-34.415933,-6.912001,-0.026716,-10.003046,58493.0,57671.221741,-3.397838,-0.699534,60.410866
8,S27,-0.012709,0.974928,45892.0,49529.070171,16.977869,3.197378,-0.095219,-5.059664,43701.0,50785.001747,33.886988,6.030218,56.368115
9,S12,0.007502,-0.451872,43248.0,39285.721384,-19.052940,-3.483271,-0.044286,-9.087021,42384.0,38015.445977,-21.219578,-3.718708,54.925693
